# Double Machine Learning for Momentum Causal Effect

**Chapter 15: Causal Estimation**
**Docker image**: `ml4t`
**Section Reference**: See Section 15.4 for DML theory and the ETF factor application

## Purpose
**Double machine learning** (DML) estimates the effect of a momentum signal on forward
returns while adjusting for volatility and regime, on a panel of ETFs. The estimate is one
of two things this notebook is about. The other is that a panel breaks every correction
that has a notion of "nearby": the standard error, the cross-validation folds, the
permutation blocks and the temporal subsets all count in rows unless they are told not to,
and on a stacked panel a row is a different ETF rather than the next day.

## Learning Objectives
- LO1: Explain what confounding does to a factor regression, and what adjustment can and
  cannot recover
- LO2: Fit a DML estimator with EconML for a continuous treatment
- LO3: Compare the raw and adjusted estimates and read the direction of the bias
- LO4: Build folds, standard errors and permutations that count in decision times
- LO5: Read a refutation test, including one that cannot refute anything

## Cross-References
- **Upstream**: Chapter 8 (ETF momentum features)
- **Downstream**: Chapter 16 (strategy simulation), Chapter 19 (risk management)
- **Related**: [`02_dowhy_causal_graph`](02_dowhy_causal_graph.ipynb) (graphical approach), [`04_dml_crypto_regime`](04_dml_crypto_regime.ipynb) (regime effects)


<!-- ML4T_ZH_BEGIN c0000-p000 -->
# 以雙重機器學習估計動能的因果效應

**第 15 章：因果估計**
**Docker 映像檔**：`ml4t`
**章節參照**：DML 理論與 ETF 因子應用見第 15.4 節

## 目的
**雙重機器學習**（DML）在 ETF 面板上，調整波動率與市場狀態後，估計動能訊號對未來報酬率的效應。本 Notebook 的兩項重點之一是估計值；另一項是面板資料會使所有帶有「鄰近」概念的校正方式出錯：標準誤、交叉驗證折、置換區塊與時間子樣本，若未明確指定，都會以資料列計數。但在堆疊面板中，相鄰資料列通常是另一檔 ETF，而非下一個交易日。

## 學習目標
- LO1：說明混雜如何影響因子迴歸，以及調整能與不能還原什麼
- LO2：使用 EconML 為連續處置擬合 DML 估計器
- LO3：比較原始與調整後估計值，解讀偏差方向
- LO4：建立以決策時點計數的驗證折、標準誤與置換程序
- LO5：解讀反駁檢定，包括一種無法反駁任何結果的檢定

## 交叉參照
- **上游**：第 8 章（ETF 動能特徵）
- **下游**：第 16 章（策略模擬）、第 19 章（風險管理）
- **相關**：[`02_dowhy_causal_graph`](02_dowhy_causal_graph.ipynb)（因果圖方法）、[`04_dml_crypto_regime`](04_dml_crypto_regime.ipynb)（市場狀態效應）
<!-- ML4T_ZH_END c0000-p000 -->

## Methodological Notes
Following Chernozhukov et al. (2017) and de Prado (2018):
- **WalkForwardCV** from ml4t-diagnostics, with purging and embargo, built over the
  panel's decision times
- **Driscoll-Kraay standard errors**, which aggregate by decision time before applying the
  Newey-West kernel
- **Block permutation within entity**, so the placebo treatment keeps its persistence

**Prerequisites**: [`02_dowhy_causal_graph`](02_dowhy_causal_graph.ipynb) for the DoWhy
workflow, and an ETF modeling dataset built by the features pipeline

## Causal Design Contract


<!-- ML4T_ZH_BEGIN c0000-p001 -->
## 方法注意事項
依據 Chernozhukov 等人（2017）與 de Prado（2018）：
- 由 ml4t-diagnostics 提供的 **WalkForwardCV**，在面板決策時點上建立，並加入清除重疊資料與禁運間隔
- **Driscoll–Kraay 標準誤**，先依決策時點彙總，再套用 Newey–West 核函數
- **在各實體內進行區塊置換**，使安慰劑處置保留其持續性

**先備條件**：閱讀 [`02_dowhy_causal_graph`](02_dowhy_causal_graph.ipynb) 了解 DoWhy 流程，並具備由特徵流程建立的 ETF 模型資料集

## 因果研究設計規約
<!-- ML4T_ZH_END c0000-p001 -->

| Element                   | This notebook                                                                                |
|---------------------------|----------------------------------------------------------------------------------------------|
| Unit                      | ETF-date row from the ETF modeling panel                                                     |
| Treatment                 | `skip_recent_6_1` - 6/1 momentum factor (continuous)                                          |
| Outcome                   | `fwd_ret_21d` - 21-day forward return                                                         |
| Controls (W in EconML)    | `vol_21d`, `vol_126d`, `regime`, `yield_curve_slope` - all backward-looking, pre-treatment    |
| Effect modifiers (X)      | None in this notebook; constant ATE target. See `04_dml_crypto_regime` for the X-slot example |
| Identification assumption | Selection on observables given the four controls; sufficient pre-treatment information       |
| Main failure mode         | Unobserved confounding (sentiment / macro shocks), nuisance-model misspecification, panel-time leakage if CV is not date-grouped |
| Estimand                  | Marginal effect of a one-unit change in 6/1 momentum on 21-day forward return after adjustment |

<!-- ML4T_ZH_BEGIN c0000-p002 -->
| 要素 | 本 Notebook |
|---|---|
| 分析單位 | ETF 模型面板中的 ETF－日期資料列 |
| 處置 | `skip_recent_6_1`，6/1 動能因子（連續） |
| 結果 | `fwd_ret_21d`，未來 21 日報酬率 |
| 控制變數（EconML 中的 W） | `vol_21d`、`vol_126d`、`regime`、`yield_curve_slope`；皆為回顧過去且發生於處置前的變數 |
| 效應調節變數（X） | 本 Notebook 無；目標為固定 ATE。X 位置的範例見 `04_dml_crypto_regime` |
| 識別假設 | 給定四項控制變數後依可觀測變數選擇；處置前資訊足夠 |
| 主要失效模式 | 未觀測混雜（情緒／總體經濟衝擊）、輔助模型設定錯誤、交叉驗證若未依日期分組而造成的面板時點資料洩漏 |
| 目標估計量 | 調整後 6/1 動能變動一單位，對未來 21 日報酬率的邊際效應 |
<!-- ML4T_ZH_END c0000-p002 -->


## Setup

<!-- ML4T_ZH_BEGIN c0001-p000 -->
## 設定
<!-- ML4T_ZH_END c0001-p000 -->


In [1]:
"""Double Machine Learning for Momentum Causal Effect - estimate causal effect of momentum on forward returns."""

import warnings

import numpy as np
import pandas as pd
from ml4t.diagnostic.splitters import WalkForwardCV
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.linear_model import Ridge

from case_studies.utils.causal import (
    block_permute,
    empirical_permutation_p,
    manual_dml_timeseries,
)
from utils.modeling import load_modeling_dataset
from utils.reproducibility import set_global_seeds

# scikit-learn repeats a notice, once per nuisance fit, that a frame carrying feature names
# was fitted and a bare array predicted; EconML does that internally. Convergence and
# numerical warnings stay visible.
warnings.filterwarnings("ignore", category=UserWarning, module="sklearn.utils.validation")

# Statsmodels for HAC standard errors
import statsmodels.api as sm
from econml.dml import LinearDML
from statsmodels.regression.linear_model import OLS

## Configuration

<!-- ML4T_ZH_BEGIN c0003-p000 -->
## 設定參數
<!-- ML4T_ZH_END c0003-p000 -->


In [2]:
# Configuration - readers can modify these
CASE_STUDY_ID = "etfs"
PRIMARY_LABEL = "fwd_ret_21d"
MAX_SYMBOLS = 0
CV_FOLDS = 5
MAX_SAMPLES = 50000  # Temporal subsample if dataset too large
SEED = 42

# Cross-validation parameters for WalkForwardCV
FORWARD_HORIZON = 21  # 21-day forward returns
LABEL_HORIZON = FORWARD_HORIZON  # Purge overlapping samples
EMBARGO_PCT = 0.01  # 1% embargo after test set

# Refutation test parameters
N_PLACEBO_PERMUTATIONS = 100
BLOCK_SIZE = 21  # Block size for permutation (match forward horizon)

In [3]:
set_global_seeds(SEED)
print(f"Seed: {SEED}")

Seed: 42


## 1. Load ETF Features from Modeling Pipeline

We use `load_modeling_dataset()` to load pre-computed features (Ch8),
temporal features (Ch9), and labels, joined and ready for analysis.
Real-data only - no synthetic fallback. If the modeling dataset is missing,
the notebook fails loudly with a clear error rather than silently switching
to a synthetic substitute that would publish indistinguishable numbers.

<!-- ML4T_ZH_BEGIN c0006-p000 -->
## 1. 從模型流程載入 ETF 特徵

我們使用 `load_modeling_dataset()` 載入預先計算的特徵（第 8 章）、時間特徵（第 9 章）及標籤；資料已連接，可直接分析。這裡只使用真實資料，不提供合成資料替代。若模型資料集缺失，Notebook 會明確報錯，而非悄悄切換至可能產生難以區分數值的合成替代資料。
<!-- ML4T_ZH_END c0006-p000 -->


In [4]:
# A load failure is fatal rather than a fallback to synthetic data.
mds = load_modeling_dataset(CASE_STUDY_ID, PRIMARY_LABEL, max_symbols=MAX_SYMBOLS)

treatment_col = "skip_recent_6_1"
outcome_col = mds.label_col
confounder_cols = ["vol_21d", "vol_126d", "regime", "yield_curve_slope"]
entity_col = mds.entity_cols[0]

available = set(mds.dataset.columns)
missing = [c for c in [treatment_col, outcome_col] + confounder_cols if c not in available]
if missing:
    raise RuntimeError(
        f"Required columns missing from modeling dataset "
        f"{CASE_STUDY_ID}/{PRIMARY_LABEL}: {missing}. "
        f"Available features: {mds.feature_names[:20]}... "
        f"Set ML4T_DATA_PATH and rebuild the Ch8 features pipeline for case "
        f"study '{CASE_STUDY_ID}'."
    )

# Convert to pandas for sklearn/econml, sorted by date
analysis_cols = [mds.date_col] + mds.entity_cols + [treatment_col, outcome_col] + confounder_cols
df = (
    mds.dataset.select([c for c in analysis_cols if c in available])
    .drop_nulls()
    .sort(mds.date_col)
    .to_pandas()
)
# Temporal subsample if too large: take the most recent N unique dates so the
# subsample never cuts through a cross-section. `df.iloc[-MAX_SAMPLES:]` would
# slice at row level on a stacked panel and leave a fragmented final date.
if len(df) > MAX_SAMPLES:
    rows_per_date = df.groupby(mds.date_col).size().median()
    n_dates = int(np.ceil(MAX_SAMPLES / max(rows_per_date, 1)))
    keep_dates = df[mds.date_col].drop_duplicates().iloc[-n_dates:]
    df = df[df[mds.date_col].isin(keep_dates)].reset_index(drop=True)
    print(f"Taking most recent {n_dates} dates ({len(df):,} rows) from {len(keep_dates):,} dates")

print(f"Analysis data: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"Date range: {df[mds.date_col].min()} to {df[mds.date_col].max()}")
print(f"Treatment: {treatment_col}, Outcome: {outcome_col}")

Taking most recent 556 dates (52,162 rows) from 556 dates
Analysis data: 52,162 rows × 8 columns
Date range: 2023-09-14 00:00:00 to 2025-12-01 00:00:00
Treatment: skip_recent_6_1, Outcome: fwd_ret_21d


### The Panel Keys, and Why Everything Below Needs Them

A row here is an ETF *and* a date, and the frame is sorted by date, so consecutive rows
are usually different ETFs on the same day rather than the same ETF on consecutive days.
Every correction in this notebook that has a notion of "nearby" - the standard error, the
cross-validation folds, the permutation blocks, the temporal subsets - has to count in
decision times, not in rows. Counting in rows on this frame measures a slice of one day's
cross-section and calls it three weeks.

The two arrays below carry that information, and each of those four places takes them.

<!-- ML4T_ZH_BEGIN c0008-p000 -->
### 面板鍵值，以及下文為何都需要它們

此處一筆資料列同時對應一檔 ETF **與**一個日期；資料框按日期排序，因此相鄰資料列通常是同日不同 ETF，而非同一 ETF 的連續兩日。本 Notebook 凡是涉及「鄰近」的校正——標準誤、交叉驗證折、置換區塊與時間子樣本——都必須以決策時點計數，而非以資料列計數。在此資料框上以資料列計數，會將某一天橫斷面的一小段誤當成三週。

下方兩個陣列承載這項資訊，上述四種程序都會使用它們。
<!-- ML4T_ZH_END c0008-p000 -->


In [5]:
decision_times = df[mds.date_col].to_numpy()
entities = df[entity_col].to_numpy()

unique_dates = np.sort(pd.unique(decision_times))
date_position = pd.Series(np.arange(len(unique_dates)), index=unique_dates)
row_date_position = date_position.reindex(decision_times).to_numpy()

print(
    f"{len(df):,} rows over {len(unique_dates):,} decision times "
    f"and {df[entity_col].nunique()} entities "
    f"({len(df) / len(unique_dates):.0f} rows per decision time)"
)


def panel_folds(n_splits):
    """Walk-forward folds built over decision times, then expanded to panel rows.

    WalkForwardCV counts `label_horizon` and the embargo in the positions it is handed. Fed
    the panel's rows it would purge a fraction of one date; fed the ordered unique dates it
    purges the 21 trading days the forward return actually spans. The row indices come back
    by membership, so no fold boundary cuts through a cross-section.
    """
    splitter = WalkForwardCV(
        n_splits=n_splits,
        label_horizon=LABEL_HORIZON,
        embargo_pct=EMBARGO_PCT,
        expanding=True,
    )
    folds = []
    for train_dates, test_dates in splitter.split(np.arange(len(unique_dates)).reshape(-1, 1)):
        folds.append(
            (
                np.flatnonzero(np.isin(row_date_position, train_dates)),
                np.flatnonzero(np.isin(row_date_position, test_dates)),
            )
        )
    return folds

52,162 rows over 556 decision times and 97 entities (94 rows per decision time)


## 2. The Unadjusted Slope

An OLS regression of forward returns on momentum, with nothing else in it. It is the
benchmark the adjusted estimates are read against, and it is confounded by construction:
volatility and the yield-curve regime both move the momentum signal and move forward
returns, so its slope carries their contribution as well as momentum's. Which way that
pushes the slope is an empirical question the comparison below answers.

<!-- ML4T_ZH_BEGIN c0010-p000 -->
## 2. 未調整的斜率

僅以動能解釋未來報酬率的 OLS 迴歸，是解讀調整後估計值的比較基準。此迴歸依建構方式受到混雜：波動率與殖利率曲線狀態都會同時影響動能訊號與未來報酬率，因此斜率除了動能效應，還包含它們的貢獻。這些因素將斜率推往哪個方向，須由下方實證比較回答。
<!-- ML4T_ZH_END c0010-p000 -->


### Which Robust Standard Error a Panel Takes

Overlapping 21-day returns make consecutive observations of one ETF correlated, and the
usual answer is a Newey-West standard error with the bandwidth set to the label horizon.
On this frame that answer is applied to the wrong axis. `cov_type="HAC"` runs its kernel
down the rows, and 21 rows here are a fifth of one day's cross-section, so the correction
treats different ETFs on the same day as if they were successive days.

**Driscoll-Kraay** is the version that fits a panel. It aggregates the regression score by
decision time first and applies the Newey-West kernel to that time series, which makes it
robust both to the serial correlation the overlap creates and to whatever the ETFs share
on a given day. statsmodels reaches it through `cov_type="hac-groupsum"` with a `time`
argument, which is the same call `case_studies/utils/causal.py` makes for every case study.

<!-- ML4T_ZH_BEGIN c0011-p000 -->
### 面板資料應使用哪種穩健標準誤

重疊的 21 日報酬率使同一檔 ETF 的相鄰觀測值相關，常見做法是使用帶寬設為標籤期間的 Newey–West 標準誤。但在此資料框中，該方法會套用到錯誤的軸。`cov_type="HAC"` 會沿資料列套用核函數，而這裡的 21 列僅相當於某一天橫斷面的約五分之一；於是校正會把同日不同 ETF 當成連續日期。

**Driscoll–Kraay** 是適用於面板的版本。它先依決策時點彙總迴歸分數，再將 Newey–West 核函數套用於所得時間序列，因此既能處理重疊造成的序列相關，也能處理 ETF 在同一天共享的因素。statsmodels 透過 `cov_type="hac-groupsum"` 並傳入 `time` 參數來執行；這也是 `case_studies/utils/causal.py` 對每個案例研究使用的呼叫方式。
<!-- ML4T_ZH_END c0011-p000 -->


In [6]:
y = df[outcome_col].values
X_with_const = sm.add_constant(df[[treatment_col]])

ols_iid = OLS(y, X_with_const).fit()
naive_estimate = float(ols_iid.params.iloc[1])
se_iid = float(ols_iid.bse.iloc[1])

# Driscoll-Kraay: the kernel runs over decision times, with the bandwidth at the label
# horizon because a 21-day forward return overlaps for 20 of every 21 days.
HAC_LAGS = FORWARD_HORIZON
time_codes = pd.factorize(decision_times, sort=False)[0]
ols_dk = ols_iid.get_robustcov_results(
    cov_type="hac-groupsum",
    time=time_codes,
    maxlags=HAC_LAGS,
    use_correction="hac",
    df_correction=False,
)
se_hac = float(np.sqrt(np.asarray(ols_dk.cov_params())[1, 1]))
t_stat_hac = naive_estimate / se_hac

naive_ci = (naive_estimate - 1.96 * se_hac, naive_estimate + 1.96 * se_hac)

print("=" * 60)
print("NAIVE ESTIMATE (ignoring confounders)")
print("=" * 60)
print(f"Coefficient: {naive_estimate:.6f}")
print(f"Standard Error (IID): {se_iid:.6f}")
print(f"Standard Error (Driscoll-Kraay): {se_hac:.6f}")
print(f"SE Inflation (DK/IID): {se_hac / se_iid:.2f}x")
print(f"95% CI (Driscoll-Kraay): [{naive_ci[0]:.6f}, {naive_ci[1]:.6f}]")
print(f"t-statistic (Driscoll-Kraay): {t_stat_hac:.2f}")

NAIVE ESTIMATE (ignoring confounders)
Coefficient: -0.035896
Standard Error (IID): 0.002104
Standard Error (Driscoll-Kraay): 0.027912
SE Inflation (DK/IID): 13.26x
95% CI (Driscoll-Kraay): [-0.090603, 0.018811]
t-statistic (Driscoll-Kraay): -1.29


## 3. Double Machine Learning Estimate under Observed-Confounder Adjustment

DML uses a three-step orthogonalization process:

1. **Predict outcome from confounders**: $\hat{Y} = g(X)$ → residual $\tilde{Y} = Y - \hat{Y}$
2. **Predict treatment from confounders**: $\hat{T} = m(X)$ → residual $\tilde{T} = T - \hat{T}$
3. **Regress residual outcome on residual treatment**: $\tilde{Y} \sim \theta \tilde{T}$

The coefficient $\theta$ is an *orthogonalized* estimate of the treatment
effect under the maintained assumption that the specified controls capture
the relevant pre-treatment confounding variation. DML reduces sensitivity
to nuisance-model errors but does not, by itself, solve unobserved
confounding, simultaneity, interference, or bad-control bias.

The folds come from `WalkForwardCV` rather than `KFold` or `TimeSeriesSplit`, because
cross-fitting a forward-looking label needs two things neither of those provides. **Purging**
drops training rows whose 21-day label window overlaps the test window; the **embargo**
leaves a gap after the test window so autocorrelation does not carry back into the next
training set (Chernozhukov et al. 2017, de Prado 2018).

<!-- ML4T_ZH_BEGIN c0013-p000 -->
## 3. 調整可觀測混雜因子後的雙重機器學習估計值

DML 採用三步驟正交化程序：

1. **由混雜因子預測結果**：$\hat{Y} = g(X)$ → 殘差 $\tilde{Y} = Y - \hat{Y}$
2. **由混雜因子預測處置**：$\hat{T} = m(X)$ → 殘差 $\tilde{T} = T - \hat{T}$
3. **以結果殘差對處置殘差進行迴歸**：$\tilde{Y} \sim \theta \tilde{T}$

係數 $\theta$ 是在指定控制變數涵蓋相關處置前混雜變異的維持假設下，對處置效應所做的**正交化**估計。DML 可降低對輔助模型誤差的敏感度，但本身無法解決未觀測混雜、同時性、個體間干擾或不當控制變數造成的偏差。

驗證折來自 `WalkForwardCV`，而非 `KFold` 或 `TimeSeriesSplit`，因為對前瞻標籤進行交叉擬合，需要後兩者均未提供的兩種措施。**清除重疊資料**會剔除 21 日標籤視窗與測試視窗重疊的訓練資料列；**禁運間隔**則在測試視窗後留出空檔，避免自我相關延續至下一訓練集（Chernozhukov 等人，2017；de Prado，2018）。
<!-- ML4T_ZH_END c0013-p000 -->


The confounders go into EconML's `W` slot, which holds controls used for residualization.
The `X` slot is for effect modifiers, the variables along which the treatment effect is
allowed to vary. Putting plain confounders in `X` still produces a single ATE, so nothing
visibly breaks, but it teaches the wrong habit for `04_dml_crypto_regime`, where the two
slots carry different variables and the distinction decides what the model estimates.

<!-- ML4T_ZH_BEGIN c0014-p000 -->
混雜因子放入 EconML 的 `W` 欄位，該欄位接受用於殘差化的控制變數。`X` 欄位則供效應調節變數使用，也就是容許處置效應隨之變動的變數。將一般混雜因子放入 `X` 仍能產生單一 ATE，表面上看不出錯誤；但對於 `04_dml_crypto_regime`，兩個欄位承載不同變數，這項區別會決定模型估計的是什麼，因此前述做法會養成錯誤習慣。
<!-- ML4T_ZH_END c0014-p000 -->


In [7]:
Y = df[outcome_col].to_numpy()
T = df[treatment_col].to_numpy()
W = df[confounder_cols].to_numpy()  # controls used for residualization

# Walk-forward folds over decision times, expanded to rows (see `panel_folds` above).
cv = panel_folds(CV_FOLDS)
print(
    f"Using WalkForwardCV over decision times (label_horizon={LABEL_HORIZON} trading days, "
    f"embargo={EMBARGO_PCT:.1%})"
)
for i, (train_idx, test_idx) in enumerate(cv):
    print(f"  fold {i}: train {len(train_idx):,} rows, test {len(test_idx):,} rows")

dml = LinearDML(
    model_y=GradientBoostingRegressor(n_estimators=50, max_depth=3, random_state=SEED),
    model_t=GradientBoostingRegressor(n_estimators=50, max_depth=3, random_state=SEED),
    cv=cv,
    random_state=SEED,
)

dml.fit(Y, T, W=W)

dml_estimate = float(dml.ate())
dml_ci_lower, dml_ci_upper = (float(v) for v in dml.ate_interval(alpha=0.05))
bias = naive_estimate - dml_estimate
bias_pct = 100 * bias / abs(dml_estimate) if dml_estimate != 0 else float("inf")

Using WalkForwardCV over decision times (label_horizon=21 trading days, embargo=1.0%)
  fold 0: train 6,816 rows, test 8,556 rows
  fold 1: train 15,384 rows, test 8,556 rows
  fold 2: train 23,940 rows, test 8,597 rows
  fold 3: train 32,516 rows, test 8,648 rows
  fold 4: train 41,164 rows, test 9,024 rows


In [8]:
print("\n" + "=" * 60)
print("DOUBLE ML ESTIMATE (controlling for confounders)")
print("=" * 60)
print(f"Average Treatment Effect: {dml_estimate:.6f}")
if not np.isnan(dml_ci_lower):
    print(f"95% CI: [{dml_ci_lower:.6f}, {dml_ci_upper:.6f}]")
else:
    print(
        "95% CI: unavailable (EconML inference returned NaN - "
        "likely numerically degenerate first-stage residualization)"
    )

print("\n" + "=" * 60)
print("COMPARISON: Naive vs DML")
print("=" * 60)
print(f"Naive estimate: {naive_estimate:.6f}")
print(f"DML estimate:   {dml_estimate:.6f}")
print(f"Bias (Naive - DML): {bias:.6f}")
print(f"Bias percentage: {bias_pct:.1f}%")

if abs(naive_estimate) > abs(dml_estimate):
    print("\n-> The adjusted effect is smaller: the controls were inflating the raw slope")
else:
    print("\n-> The adjusted effect is larger: the controls were masking part of the slope")


DOUBLE ML ESTIMATE (controlling for confounders)
Average Treatment Effect: -0.058517
95% CI: [-0.065232, -0.051802]

COMPARISON: Naive vs DML
Naive estimate: -0.035896
DML estimate:   -0.058517
Bias (Naive - DML): 0.022621
Bias percentage: 38.7%

-> The adjusted effect is larger: the controls were masking part of the slope


**Interpretation**: The naive OLS estimate captures both the causal momentum effect and
spurious correlation induced by shared confounders (volatility, regime). DML's
orthogonalization removes this confounding by residualizing both treatment and outcome
against the confounders before estimating the final coefficient.

The direction of bias reveals the confounding structure. If the naive effect is
smaller in magnitude than the DML estimate, confounders *mask* the true effect -
for example, high volatility reduces both momentum and returns simultaneously.
If the naive effect is larger, confounders *inflate* the apparent predictive power.

**What it changes downstream**: a position size derived from the raw slope inherits
whatever the confounders contributed to it. The adjusted estimate is the one to size on
when volatility and regime are not separately hedged, and the gap between the two is how
much the raw slope was carrying.

<!-- ML4T_ZH_BEGIN c0017-p000 -->
**解讀**：天真的 OLS 估計值同時包含動能的因果效應，以及共同混雜因子（波動率、市場狀態）誘發的虛假相關。DML 在估計最終係數前，將處置與結果分別對混雜因子殘差化，以正交化程序移除這些混雜影響。

偏差方向揭示混雜結構。若天真估計的效應幅度小於 DML 估計值，混雜因子便**掩蓋**了真實效應，例如高波動率同時降低動能與報酬率。若天真估計的效應較大，混雜因子便**誇大**表面上的預測能力。

**對下游的影響**：若依原始斜率決定部位大小，部位會承接混雜因子對斜率的貢獻。當波動率與市場狀態未另外避險時，應依調整後估計值決定部位大小；兩個估計值的差距顯示原始斜率承載了多少混雜影響。
<!-- ML4T_ZH_END c0017-p000 -->


## 4. The Same Estimate, Assembled by Hand

`manual_dml_timeseries` from `case_studies/utils/causal.py` runs the three steps in the
open: cross-fit the two nuisance models over walk-forward folds with an embargo, take the
residuals, regress one on the other. Handed `groups`, it builds those folds over decision
times and reports a Driscoll-Kraay standard error; without them it would do both by row.

<!-- ML4T_ZH_BEGIN c0018-p000 -->
## 4. 手動組合出相同估計值

`manual_dml_timeseries` 來自 `case_studies/utils/causal.py`，將三個步驟明確展開：在帶禁運間隔的滾動向前驗證折上交叉擬合兩個輔助模型，取得殘差，再以其中一組殘差對另一組迴歸。傳入 `groups` 時，它會依決策時點建立驗證折並報告 Driscoll–Kraay 標準誤；若未傳入，兩者都會以資料列為單位。
<!-- ML4T_ZH_END c0018-p000 -->


In [9]:
# Use shared manual DML implementation
dml_result = manual_dml_timeseries(
    df[outcome_col].values,
    df[treatment_col].values,
    df[confounder_cols].values,
    n_folds=CV_FOLDS,
    embargo=LABEL_HORIZON,
    groups=decision_times,
    horizon=FORWARD_HORIZON,
)

manual_ate = dml_result["theta"]
manual_se_iid = dml_result["se_iid"]
manual_se_hac = dml_result["se_hac"]
manual_t_hac = dml_result["t_stat_hac"]
manual_ci = (manual_ate - 1.96 * manual_se_hac, manual_ate + 1.96 * manual_se_hac)

print("\n" + "=" * 60)
print("MANUAL DML WITH WALK-FORWARD CV + EMBARGO")
print("=" * 60)
print(f"ATE estimate: {manual_ate:.6f}")
print(f"Standard Error (IID): {manual_se_iid:.6f}")
print(f"Standard Error (HAC): {manual_se_hac:.6f}")
print(f"SE Inflation (HAC/IID): {manual_se_hac / manual_se_iid:.2f}x")
print(f"95% CI (HAC): [{manual_ci[0]:.6f}, {manual_ci[1]:.6f}]")
print(f"t-statistic (Driscoll-Kraay): {manual_t_hac:.2f}")


MANUAL DML WITH WALK-FORWARD CV + EMBARGO
ATE estimate: -0.049251
Standard Error (IID): 0.004014
Standard Error (HAC): 0.026538
SE Inflation (HAC/IID): 6.61x
95% CI (HAC): [-0.101264, 0.002763]
t-statistic (Driscoll-Kraay): -1.86


## 5. Refutation Tests

**Critical for causal validity**: We validate the DML estimate using refutation tests.

1. **Temporal Placebo**: Regress Y on *lead* of T (should be ~0 if no reverse causality)
2. **Block Permutation**: Shuffle treatment in blocks to preserve autocorrelation
3. **Subset Stability**: Check if effect is stable across temporal subsets

<!-- ML4T_ZH_BEGIN c0020-p000 -->
## 5. 反駁檢定

**對因果有效性至關重要**：我們使用反駁檢定驗證 DML 估計值。

1. **時間安慰劑**：以 T 的**領先值**對 Y 進行迴歸（若不存在反向因果，結果應約為零）
2. **區塊置換**：以區塊打亂處置，以保留自我相關
3. **子樣本穩定性**：檢查效應在不同時間子樣本間是否穩定
<!-- ML4T_ZH_END c0020-p000 -->


In [10]:
print("\n" + "=" * 60)
print("REFUTATION TESTS")
print("=" * 60)

# The lead is taken within symbol; a row-level shift would mix ETFs at the boundary.
print("\n1. TEMPORAL PLACEBO TEST (lead of treatment)")
df_placebo = df.sort_values([entity_col, mds.date_col]).copy()
df_placebo["treatment_lead"] = df_placebo.groupby(entity_col)[treatment_col].shift(-FORWARD_HORIZON)
# Back to date order before the fit. The shift needed entity-major rows; the folds need
# every row of a decision time adjacent, and `manual_dml_timeseries` rejects groups that
# are not sorted and contiguous rather than silently splitting one date across two folds.
df_placebo = (
    df_placebo.dropna(subset=["treatment_lead", outcome_col])
    .sort_values(mds.date_col, kind="stable")
    .reset_index(drop=True)
)

if len(df_placebo) > 100:
    placebo_result = manual_dml_timeseries(
        df_placebo[outcome_col].values,
        df_placebo["treatment_lead"].values,
        df_placebo[confounder_cols].values,
        n_folds=CV_FOLDS,
        embargo=LABEL_HORIZON,
        groups=df_placebo[mds.date_col].to_numpy(),
        horizon=FORWARD_HORIZON,
    )
    placebo_effect = placebo_result["theta"]

    print(f"   Lead treatment effect (DML): {placebo_effect:.6f}")
    print(f"   Original DML effect:         {manual_ate:.6f}")
    print(f"   Ratio |lead / original|:     {abs(placebo_effect) / abs(manual_ate):.3f}")
else:
    placebo_effect = None
    print("   Insufficient data for placebo test")


REFUTATION TESTS

1. TEMPORAL PLACEBO TEST (lead of treatment)


   Lead treatment effect (DML): -0.075183
   Original DML effect:         -0.049251
   Ratio |lead / original|:     1.527


**Reading the temporal placebo.** A ratio near one is what this test returns here, and it
is not evidence of reverse causality. The treatment is a momentum signal that barely moves
over 21 trading days, so the horizon-shifted placebo is nearly the same variable as the
real treatment and reproduces a similar effect. For a treatment this persistent, the
shifted-signal placebo cannot refute anything.

The block permutation below is the refuter that can, because it breaks the alignment
between treatment and outcome while keeping the treatment's own persistence intact - but
only if the blocks are built along time.

<!-- ML4T_ZH_BEGIN c0022-p000 -->
**解讀時間安慰劑。** 此檢定在這裡得到接近一的比率，但這不是反向因果的證據。處置是動能訊號，在 21 個交易日內幾乎不變，因此將訊號按預測期間平移後的安慰劑，與真實處置幾乎是同一變數，也會重現近似的效應。對持續性如此高的處置，平移訊號的安慰劑無法反駁任何結果。

下方的區塊置換則有反駁能力，因為它在保留處置自身持續性的同時，打破處置與結果的對齊；但前提是區塊沿時間建立。
<!-- ML4T_ZH_END c0022-p000 -->


### Blocks Along Time, Not Along Rows

`block_permute` given a bare array counts `BLOCK_SIZE` in the positions it is handed. On
this frame that is about a fifth of one day's cross-section, so a "block" is a handful of
ETFs on a single day. Permuting those destroys the serial dependence the test is supposed
to preserve, which is the iid shuffle `block_permute`'s own docstring warns makes placebo
tests too easy to pass.

Passing `groups` and `units` makes it permute within each ETF, where `BLOCK_SIZE` counts
that ETF's own ordered trading days and a block is the three weeks the name promises.

Each placebo also runs the estimator being tested, with the same fold count and embargo. A
placebo fitted on three folds against an estimate fitted on five is a different estimator
on a different number of cross-fitted rows, and the null would then be centred wherever
that difference puts it rather than where the absence of an effect does. The observed
statistic itself moves between the two settings.

<!-- ML4T_ZH_BEGIN c0023-p000 -->
### 區塊須沿時間建立，而非沿資料列

`block_permute` 若只收到陣列，就會在陣列位置上計算 `BLOCK_SIZE`。在此資料框上，這約等於某一天橫斷面的五分之一，因此一個「區塊」只是同日的一小群 ETF。置換這些資料會破壞檢定原本要保留的序列相依性，形成 `block_permute` 自身文件字串警告的獨立同分布打亂方式，讓安慰劑檢定太容易通過。

傳入 `groups` 與 `units`，可讓置換在各 ETF 內進行；此時 `BLOCK_SIZE` 會以該 ETF 自身按順序排列的交易日計數，一個區塊才真正涵蓋名稱所指的三週。

每個安慰劑也會執行受檢驗的估計器，使用相同驗證折數與禁運間隔。若安慰劑使用三折，原估計值卻使用五折，兩者就成了在不同數量交叉擬合資料列上執行的不同估計器；虛無分布中心便會反映這項差異，而非效應不存在時的位置。兩種設定下，觀測統計量本身也會改變。
<!-- ML4T_ZH_END c0023-p000 -->


In [11]:
# Test 2: Block Permutation Test (uses shared block_permute)
print(f"\n2. BLOCK PERMUTATION TEST ({N_PLACEBO_PERMUTATIONS} permutations)")
placebo_effects = []
placebo_t_stats = []
permutation_failures = 0
T_original = df[treatment_col].values
rng = np.random.default_rng(SEED)

for i in range(N_PLACEBO_PERMUTATIONS):
    T_permuted = block_permute(
        T_original,
        BLOCK_SIZE,
        rng=rng,
        groups=decision_times,
        units=entities,
    )

    df_perm = df.copy()
    df_perm[treatment_col] = T_permuted

    try:
        perm_result = manual_dml_timeseries(
            df_perm[outcome_col].values,
            df_perm[treatment_col].values,
            df_perm[confounder_cols].values,
            n_folds=CV_FOLDS,  # the estimate's own setting; see the markdown above
            embargo=LABEL_HORIZON,
            groups=decision_times,
            horizon=FORWARD_HORIZON,
        )
        if np.isfinite(perm_result["t_stat_hac"]):
            placebo_t_stats.append(perm_result["t_stat_hac"])
            placebo_effects.append(perm_result["theta"])
        else:
            permutation_failures += 1
    except Exception as exc:
        permutation_failures += 1
        print(f"   Permutation {i} failed: {type(exc).__name__}: {exc}")

PERMUTATION_MIN_SUCCESS = max(10, int(0.5 * N_PLACEBO_PERMUTATIONS))
print(f"   Permutations: {len(placebo_effects)} successful, {permutation_failures} failed")
if len(placebo_effects) < PERMUTATION_MIN_SUCCESS:
    raise RuntimeError(
        f"Block-permutation placebo: only {len(placebo_effects)} successful "
        f"runs (need ≥{PERMUTATION_MIN_SUCCESS}); refuter cannot be trusted."
    )


2. BLOCK PERMUTATION TEST (100 permutations)


   Permutations: 100 successful, 0 failed


### Compared on t-Statistics, Not on Effect Sizes

Permuting the treatment also frees it from the controls. The second stage regresses the
residualized outcome on the residualized treatment, so whatever the controls explain leaves
the denominator; a permuted treatment is no longer explained by them, its residual variance
is larger, and a placebo effect is mechanically smaller than the observed one whether or not
there is anything to find. Comparing raw effects against that distribution reports
significance the standard error does not support. Each permutation's t-statistic divides by
its own standard error, so the scale cancels and only the alignment between treatment and
outcome is left.

The p-value is the fraction of the placebo distribution at least as extreme as the observed
statistic, and it carries a plus-one correction because the observed statistic is itself one
draw that distribution could produce. Without the correction, a run in which no placebo
reaches it reports zero, which no finite number of permutations can establish. With n draws
the smallest reportable value is 1 / (n + 1), printed beside it as the floor. It is not a
false discovery rate, which is what this quantity used to be called.


<!-- ML4T_ZH_BEGIN c0025-p000 -->
### 比較 t 統計量，而非效應大小

置換處置也使其脫離控制變數。第二階段以結果殘差對處置殘差迴歸，因此控制變數所解釋的部分會從分母移除；置換後的處置不再受它們解釋，殘差變異較大，安慰劑效應因而在機制上小於觀測效應，不論是否真的存在效應。若把原始效應與此分布比較，會報告標準誤無法支持的顯著性。每次置換的 t 統計量除以自身標準誤，使尺度抵銷，只留下處置與結果的對齊關係。

p 值是安慰劑分布中至少與觀測統計量一樣極端的比例，並採加一校正，因為觀測統計量本身也是該分布可能產生的一次抽樣。若不校正，當沒有安慰劑達到觀測值時，結果會報告為零，但有限次置換無法確立零。若抽樣 n 次，最小可報告值為 1／（n＋1），並在旁列為下限。它不是此量過去曾被誤稱的偽發現率。
<!-- ML4T_ZH_END c0025-p000 -->

**What cancelling the scale does not buy.** It removes one known bias and does not make the
test calibrated. A permutation test is valid when the permuted labels are exchangeable under
the null, and a treatment that persistent confounders predict is not: the shuffle breaks the
confounding along with the effect, so an estimate that is itself biased sits far from its own
permutation null and the test reports that distance. Measured on twelve synthetic panels with
a true effect of exactly zero and persistent AR(1) confounders, the studentized comparison
still rejects at the conventional five percent level on five of the twelve, against eleven for
the raw-effect comparison. `10_case_study_insights` carries that measurement. Read the count below as a comparison against a shuffle
rather than as a significance level.

<!-- ML4T_ZH_BEGIN c0025-p001 -->
**消除尺度差異仍有其限制。** 這只移除一項已知偏差，並未使檢定獲得校準。置換檢定在虛無假設下，被置換的標籤具有可交換性時才有效；若持續性混雜因子可以預測處置，就不符合此條件。打亂會連同效應一起破壞混雜，因此本身帶偏差的估計值會遠離自己的置換虛無分布，而檢定報告的就是這段距離。在真實效應恰為零、且混雜因子具有持續性 AR(1) 結構的十二份合成面板上，標準化比較仍在十二次中有五次於慣用的百分之五水準拒絕；原始效應比較則有十一次。`10_case_study_insights` 記錄此測量。下方的次數應解讀為與打亂結果的比較，而非顯著水準。
<!-- ML4T_ZH_END c0025-p001 -->


In [12]:
if len(placebo_t_stats) > 10:
    placebo_mean = np.mean(placebo_t_stats)
    placebo_std = np.std(placebo_t_stats)
    z_score = (manual_t_hac - placebo_mean) / placebo_std if placebo_std > 0 else np.inf
    # The plus-one correction is why the floor below is 1 / (n + 1); see the markdown above.
    permutation_p = empirical_permutation_p(np.asarray(placebo_t_stats), manual_t_hac)

    print(f"   Placebo t mean: {placebo_mean:.4f}")
    print(f"   Placebo t std:  {placebo_std:.4f}")
    print(f"   Observed t (Driscoll-Kraay): {manual_t_hac:.4f}")
    print(f"   Z-score vs placebo: {z_score:.2f}")
    print(
        f"   Permutation p-value: {permutation_p:.4f} (floor {1 / (len(placebo_t_stats) + 1):.4f})"
    )

    print(
        f"   Placebo draws at least as extreme: "
        f"{int(round(permutation_p * (len(placebo_t_stats) + 1))) - 1} of {len(placebo_t_stats)}"
    )
    print(
        f"   Placebo effect spread {np.std(placebo_effects):.6f} against a Driscoll-Kraay "
        f"standard error of {manual_se_hac:.6f}; the comparison above is on t-statistics, "
        f"which holds whatever the ratio of those two turns out to be"
    )
else:
    print("   Insufficient successful permutations")
    z_score = None
    permutation_p = None

   Placebo t mean: 1.7379
   Placebo t std:  1.2094
   Observed t (Driscoll-Kraay): -1.8559
   Z-score vs placebo: -2.97
   Permutation p-value: 0.4752 (floor 0.0099)
   Placebo draws at least as extreme: 47 of 100
   Placebo effect spread 0.023095 against a Driscoll-Kraay standard error of 0.026538; the comparison above is on t-statistics, which holds whatever the ratio of those two turns out to be


**The z-score and the p-value can disagree, and the count is the one that holds.** The
z-score measures how far the observed t-statistic sits from the placebo *mean* in placebo
standard deviations, which is a statement about a normal distribution centred where the
placebos are. The permutation p-value counts how many placebo draws reach its magnitude.
When the placebo distribution is not centred near zero the two answer different questions,
and only the count is a statement about the null the test actually built. Read the count
printed above, and the mean and standard deviation beside it, before reading the z-score.

**This null is not centred at zero, and that is worth looking at rather than reporting.**
A permutation that implemented "no association" would put the placebo statistics around
zero. Two obvious explanations do not survive a check: demeaning the fitted residuals by
symbol leaves the placebo estimates positive, and so does demeaning them by date, so
neither a preserved between-symbol nor a preserved between-date component accounts for it.
What does is not settled here. Until it is, the count is the statistic to read - it asks
how often a permuted treatment reaches the observed statistic, which stays a fair question
whatever the distribution's centre - and the z-score, which measures distance from that
centre in placebo standard deviations, is a statement about a distribution the test has not
explained.

<!-- ML4T_ZH_BEGIN c0027-p000 -->
**z 分數與 p 值可能不一致，應以計數結果為準。** z 分數衡量觀測 t 統計量距離安慰劑**平均值**幾個安慰劑標準差，這是以安慰劑中心為中心的常態分布下的陳述。置換 p 值則計算有多少次安慰劑抽樣達到觀測統計量的幅度。當安慰劑分布的中心不接近零時，兩者回答不同問題；只有計數結果描述了此檢定實際建立的虛無分布。解讀 z 分數前，應先看上方列出的計數，以及旁邊的平均值與標準差。

**此虛無分布並非以零為中心，值得探查，而非直接據以報告結論。** 若置換實作的是「沒有關聯」，安慰劑統計量應分布於零附近。兩種顯而易見的解釋經檢查後不成立：依股票代號對擬合殘差去平均，安慰劑估計值仍為正；依日期去平均也是如此。因此，保留下來的跨股票或跨日期成分都無法解釋此現象。真正原因尚未在此確定。在釐清之前，應解讀計數結果：它詢問置換處置有多常達到觀測統計量，不論分布中心在哪，這仍是合理的問題；z 分數則衡量與該中心相距幾個安慰劑標準差，涉及檢定尚未解釋的分布。
<!-- ML4T_ZH_END c0027-p000 -->


The two halves are cut at a decision time rather than at a row, so neither holds a
fragment of a cross-section. That is the same reason the subsample near the top keeps
whole dates.

<!-- ML4T_ZH_BEGIN c0028-p000 -->
兩半樣本在決策時點切分，而非在資料列切分，因此都不會只包含某個橫斷面的一部分。上方的子樣本也保留完整日期，原因相同。
<!-- ML4T_ZH_END c0028-p000 -->


In [13]:
print("\n3. SUBSET STABILITY TEST (temporal halves)")
midpoint_date = unique_dates[len(unique_dates) // 2]
df_first_half = df[decision_times < midpoint_date]
df_second_half = df[decision_times >= midpoint_date]

if len(df_first_half) > 100 and len(df_second_half) > 100:
    result_first = manual_dml_timeseries(
        df_first_half[outcome_col].values,
        df_first_half[treatment_col].values,
        df_first_half[confounder_cols].values,
        n_folds=3,
        groups=df_first_half[mds.date_col].to_numpy(),
        horizon=FORWARD_HORIZON,
    )
    result_second = manual_dml_timeseries(
        df_second_half[outcome_col].values,
        df_second_half[treatment_col].values,
        df_second_half[confounder_cols].values,
        n_folds=3,
        groups=df_second_half[mds.date_col].to_numpy(),
        horizon=FORWARD_HORIZON,
    )

    effect_first = result_first["theta"]
    se_first = result_first["se_hac"]
    effect_second = result_second["theta"]
    se_second = result_second["se_hac"]

    print(f"   First half effect:  {effect_first:.6f} (SE: {se_first:.6f})")
    print(f"   Second half effect: {effect_second:.6f} (SE: {se_second:.6f})")

    diff = abs(effect_first - effect_second)
    diff_se = np.sqrt(se_first**2 + se_second**2)
    diff_z = diff / diff_se if diff_se > 0 else 0

    print(f"   Difference: {diff:.6f} (z = {diff_z:.2f})")

    print(f"   First half: {len(df_first_half):,} rows before {midpoint_date}")
    print(f"   Second half: {len(df_second_half):,} rows from {midpoint_date}")
else:
    print("   Insufficient data for subset test")


3. SUBSET STABILITY TEST (temporal halves)


   First half effect:  -0.062793 (SE: 0.042257)
   Second half effect: -0.017201 (SE: 0.032803)
   Difference: 0.045592 (z = 0.85)
   First half: 26,079 rows before 2024-10-22T00:00:00.000
   Second half: 26,083 rows from 2024-10-22T00:00:00.000


## 6. Nuisance Model Sensitivity

DML results depend on the quality of nuisance models. Let's compare different choices.

<!-- ML4T_ZH_BEGIN c0030-p000 -->
## 6. 輔助模型敏感度

DML 結果取決於輔助模型的品質。以下比較不同選擇。
<!-- ML4T_ZH_END c0030-p000 -->


In [14]:
results = []

nuisance_models = [
    ("Linear", Ridge(alpha=1.0), Ridge(alpha=1.0)),
    (
        "GBM (shallow)",
        GradientBoostingRegressor(n_estimators=50, max_depth=2, random_state=SEED),
        GradientBoostingRegressor(n_estimators=50, max_depth=2, random_state=SEED),
    ),
    (
        "GBM (deep)",
        GradientBoostingRegressor(n_estimators=100, max_depth=5, random_state=SEED),
        GradientBoostingRegressor(n_estimators=100, max_depth=5, random_state=SEED),
    ),
]

cv_sensitivity = panel_folds(3)

for name, model_y, model_t in nuisance_models:
    dml_test = LinearDML(model_y=model_y, model_t=model_t, cv=cv_sensitivity, random_state=SEED)
    dml_test.fit(Y, T, W=W)
    ate = float(dml_test.ate())
    results.append({"Nuisance Model": name, "ATE Estimate": ate})

sensitivity_df = pd.DataFrame(results)
print("\n" + "=" * 60)
print("NUISANCE MODEL SENSITIVITY")
print("=" * 60)
print(sensitivity_df.to_string(index=False))
# A ratio of magnitudes reports 1.00x, which reads as no spread at all, when the
# specifications straddle zero symmetrically - the one case where the nuisance choice
# changes the conclusion rather than its size. The signed range cannot do that.
lo = float(sensitivity_df["ATE Estimate"].min())
hi = float(sensitivity_df["ATE Estimate"].max())
print(
    f"\nAcross {len(sensitivity_df)} nuisance specifications the estimate spans "
    f"{lo:.6f} to {hi:.6f}, a range of {hi - lo:.6f} in {outcome_col} units"
)
print(f"  Specifications agreeing in sign: {'all' if lo * hi > 0 else 'not all'}")


NUISANCE MODEL SENSITIVITY
Nuisance Model  ATE Estimate
        Linear     -0.063726
 GBM (shallow)     -0.082061
    GBM (deep)     -0.086313

Across 3 nuisance specifications the estimate spans -0.086313 to -0.063726, a range of 0.022587 in fwd_ret_21d units
  Specifications agreeing in sign: all


## 7. Results Summary

<!-- ML4T_ZH_BEGIN c0032-p000 -->
## 7. 結果摘要
<!-- ML4T_ZH_END c0032-p000 -->


In [15]:
print("\n" + "=" * 60)
print("CHAPTER 15 NOTEBOOK RESULTS: 03_econml_dml.py")
print("=" * 60)

results_summary = {
    "naive_estimate": naive_estimate,
    "naive_se_iid": se_iid,
    "naive_se_hac": se_hac,
    "naive_ci_lower": naive_ci[0],
    "naive_ci_upper": naive_ci[1],
    "manual_dml_estimate": manual_ate,
    "manual_dml_se_iid": manual_se_iid,
    "manual_dml_se_hac": manual_se_hac,
}

if dml_estimate is not None:
    results_summary["econml_dml_estimate"] = dml_estimate
    results_summary["econml_dml_ci_lower"] = dml_ci_lower
    results_summary["econml_dml_ci_upper"] = dml_ci_upper
    results_summary["bias_from_confounding"] = bias
    results_summary["bias_pct"] = bias_pct

if z_score is not None:
    results_summary["placebo_z_score"] = z_score
if permutation_p is not None:
    results_summary["permutation_p"] = permutation_p

for key, value in results_summary.items():
    if isinstance(value, float):
        print(f"{key}: {value:.6f}")
    else:
        print(f"{key}: {value}")


CHAPTER 15 NOTEBOOK RESULTS: 03_econml_dml.py
naive_estimate: -0.035896
naive_se_iid: 0.002104
naive_se_hac: 0.027912
naive_ci_lower: -0.090603
naive_ci_upper: 0.018811
manual_dml_estimate: -0.049251
manual_dml_se_iid: 0.004014
manual_dml_se_hac: 0.026538
econml_dml_estimate: -0.058517
econml_dml_ci_lower: -0.065232
econml_dml_ci_upper: -0.051802
bias_from_confounding: 0.022621
bias_pct: 38.657063
placebo_z_score: -2.971589
permutation_p: 0.475248


## Key Methodological Points

### The four choices that decide whether the estimate means anything
1. **Cross-fitting with purging and embargo**, over decision times rather than rows.
   Purging drops training rows whose labels overlap the test window; the embargo adds a
   buffer after it (Chernozhukov et al. 2017, de Prado 2018).
2. **Driscoll-Kraay standard errors**, so the serial-correlation correction is applied
   along time and the cross-sectional dependence within a date is absorbed rather than
   counted as extra observations.
3. **Block permutation within entity**, so the placebo distribution is built from a
   treatment that keeps its persistence.
4. **A nuisance-model sweep**, because the point estimate depends on the first stage and
   the spread across specifications is the honest width of the finding.

<!-- ML4T_ZH_BEGIN c0034-p000 -->
## 重要方法要點

### 決定估計值是否有意義的四項選擇
1. **依決策時點進行交叉擬合，並清除重疊資料及加入禁運間隔**，而非依資料列進行。清除步驟剔除標籤與測試視窗重疊的訓練資料列；禁運間隔在其後增加緩衝（Chernozhukov 等人，2017；de Prado，2018）。
2. **Driscoll–Kraay 標準誤**，使序列相關校正沿時間軸套用，並吸收同日橫斷面相依性，而非將其當成額外觀測值。
3. **在各實體內進行區塊置換**，讓安慰劑分布由保留持續性的處置建立。
4. **掃描多種輔助模型設定**，因為點估計值取決於第一階段模型，不同設定的結果範圍才如實呈現發現的不確定程度。
<!-- ML4T_ZH_END c0034-p000 -->


### Three Intervals on One Effect

The notebook produces three statements of uncertainty about the same quantity, and they do
not agree. EconML's `ate_interval` treats the residualized observations as independent
draws. The manual DML path reports a Driscoll-Kraay standard error, which aggregates by
decision time and absorbs whatever the ETFs share on a day. The permutation test compares
the estimate against a null built by shuffling the treatment within each ETF.

The spread between them is not a defect in any one of them. It is the price of the panel:
an interval is only as good as its account of what is independent, and 52,000 ETF-days are
not 52,000 independent observations.

<!-- ML4T_ZH_BEGIN c0035-p000 -->
### 同一效應的三種區間

本 Notebook 對同一個量提出三種不確定性陳述，結果並不一致。EconML 的 `ate_interval` 將殘差化後的觀測值視為獨立抽樣。手動 DML 流程則報告 Driscoll–Kraay 標準誤，先依決策時點彙總，並吸收 ETF 在同一天共享的因素。置換檢定則將估計值與各 ETF 內打亂處置所得的虛無分布比較。

這些結果的差異並非某一種方法的缺陷，而是面板資料的代價：區間的可靠度取決於它如何界定獨立性；52,000 個 ETF－日期不等於 52,000 筆獨立觀測值。
<!-- ML4T_ZH_END c0035-p000 -->


In [16]:
se_inflation = se_hac / se_iid

print("Quantitative Findings")
print("-" * 40)
print(f"SE inflation, Driscoll-Kraay over iid: {se_inflation:.2f}x")
if dml_estimate is not None:
    direction = "larger" if abs(naive_estimate) > abs(dml_estimate) else "smaller"
    print(
        f"Adjustment moves the slope by {abs(bias_pct):.1f}% - the unadjusted estimate is "
        f"{direction} in magnitude"
    )
    if not np.isnan(dml_ci_lower):
        print(
            f"EconML DML: {dml_estimate:.6f}, iid 95% CI "
            f"[{dml_ci_lower:.6f}, {dml_ci_upper:.6f}], width {dml_ci_upper - dml_ci_lower:.6f}"
        )
    else:
        print(f"EconML DML: {dml_estimate:.6f} (interval unavailable)")
print(
    f"Manual DML:  {manual_ate:.6f}, Driscoll-Kraay 95% CI "
    f"[{manual_ci[0]:.6f}, {manual_ci[1]:.6f}], width {manual_ci[1] - manual_ci[0]:.6f}"
)
print(
    f"Naive OLS:   {naive_estimate:.6f}, Driscoll-Kraay 95% CI [{naive_ci[0]:.6f}, {naive_ci[1]:.6f}]"
)
if z_score is not None:
    print(f"Placebo z-score (on t-statistics): {z_score:.2f}")
if permutation_p is not None:
    print(f"Permutation p-value: {permutation_p:.4f}")

Quantitative Findings
----------------------------------------
SE inflation, Driscoll-Kraay over iid: 13.26x
Adjustment moves the slope by 38.7% - the unadjusted estimate is smaller in magnitude
EconML DML: -0.058517, iid 95% CI [-0.065232, -0.051802], width 0.013430
Manual DML:  -0.049251, Driscoll-Kraay 95% CI [-0.101264, 0.002763], width 0.104027
Naive OLS:   -0.035896, Driscoll-Kraay 95% CI [-0.090603, 0.018811]
Placebo z-score (on t-statistics): -2.97
Permutation p-value: 0.4752


## Key Takeaways

1. **DML is an adjusted estimate, not a free lunch**. The orthogonalized
   coefficient is interpretable as a causal effect only under the maintained
   assumptions: pre-treatment controls are sufficient, positivity holds, no
   interference, and the specified DAG is correct.

2. **The robust standard error has to match the data's axis.** Overlapping 21-day labels
   make the iid standard error too small, and on a stacked panel the ordinary Newey-West
   correction fixes that along the wrong axis: its lags run across the cross-section.
   Driscoll-Kraay aggregates by decision time first, which is what the SE inflation
   printed above measures.

3. **Manual DML matches EconML conceptually and differs numerically.** The point estimate
   moves with nuisance-model flexibility, and the sweep in section 6 shows by how much.
   Neyman orthogonality is a guarantee about first-order sensitivity to nuisance error,
   not about agreement across nuisance choices.

4. **A refutation counts in the units its name claims.** Block permutation preserves
   autocorrelation only when the blocks run along an entity's own trading days; on a
   flattened panel the same call permutes a slice of one day's cross-section, which is
   the iid shuffle the test exists to avoid. The same is true of the walk-forward folds
   and of the temporal halves.

5. **The sign of the confounding bias is a result, not a rule.** Here the controls change
   the momentum slope in one direction; on the crypto funding premium in
   `02_dowhy_causal_graph` they change it in the other. The comparison printed above says
   which happened on this data, and it is the printed numbers, not this sentence, that
   settle it.

<!-- ML4T_ZH_BEGIN c0037-p000 -->
## 重點整理

1. **DML 是經調整的估計值，並非免費保證。** 只有在維持的假設成立時，正交化係數才能解讀為因果效應：處置前控制變數足夠、正值性成立、沒有個體間干擾，且指定的 DAG 正確。

2. **穩健標準誤必須符合資料的軸。** 重疊的 21 日標籤會使獨立同分布標準誤過小；在堆疊面板上，普通 Newey–West 校正沿錯誤的軸修正：其落後期數跨越橫斷面。Driscoll–Kraay 先依決策時點彙總，上方輸出的標準誤膨脹反映的正是此差異。

3. **手動 DML 與 EconML 在概念上相同，數值上卻不同。** 點估計值會隨輔助模型的彈性變動，第 6 節的設定掃描顯示變動幅度。Neyman 正交性保證的是對輔助模型誤差的一階敏感度，而非不同輔助模型選擇會得出相同結果。

4. **反駁檢定必須以名稱所指的單位計數。** 區塊只有沿各實體自身交易日建立，才會保留自我相關；在攤平的面板上，相同函式呼叫置換的是某一天橫斷面的一小段，正是檢定原本要避免的獨立同分布打亂方式。滾動向前驗證折與時間切半也遵循相同原則。

5. **混雜偏差的正負號是結果，不是定律。** 這裡控制變數使動能斜率朝某方向變動；在 `02_dowhy_causal_graph` 的加密資產資金費溢價分析中，斜率則朝相反方向變動。上方輸出的比較說明本資料發生了什麼；真正決定結論的是輸出數值，而非這句文字。
<!-- ML4T_ZH_END c0037-p000 -->
